In [ ]:
""" 
Notebook to publish and read data from SQS.:

    - Publishes messages to the queue.
    - Reads messages from both the standard and FIFO queues.
    - Deletes messages to prevent reprocessing.

EDEM. Master Big Data & Cloud 2025/2026
Professor: Javi Briones
"""

#### Setup

In [ ]:
# Load environment variables from .env file
from dotenv import load_dotenv
load_dotenv(dotenv_path="../../00_DocAux/.env")

In [ ]:
# Your AWS Credentials
import os

AWS_ACCESS_KEY = os.getenv("AWS_ACCESS_KEY")
AWS_SECRET_KEY = os.getenv("AWS_SECRET_KEY")
AWS_REGION = os.getenv("AWS_REGION", "eu-north-1") 

In [ ]:
queue_url = '<YOUR_STANDARD_QUEUE_URL>'
fifo_queue_url = '<YOUR_FIFO_QUEUE_URL>'

In [ ]:
import boto3

# Create session
session = boto3.Session(
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name=AWS_REGION
)

In [ ]:
# Set SQS Client
sqs = session.client('sqs')

#### Publish Messages to the Queue

In [ ]:
from datetime import datetime
import json

try:

    for i in range(5):

        response = sqs.send_message(
            QueueUrl=queue_url,
            MessageBody=json.dumps({"msg": f'My First message: {i}', "tmp": str(datetime.now())})
        )

        print("message sent ", response['MessageId'])

except Exception as e:

    print(e)

#### Publish multiple messages to verify message ordering.

In [ ]:
from datetime import datetime
import json
import uuid

try:

    for i in range(5):

        response = sqs.send_message(
            QueueUrl=fifo_queue_url,
            MessageBody=json.dumps({"msg": f'My First message: {i}', "tmp": str(datetime.now())}),
            MessageGroupId='edem-message-group',
            MessageDeduplicationId=str(uuid.uuid4())
        )

        print("message sent ", response['MessageId'])

except Exception as e:

    print(e)

#### Read Messages from the Queue

In [ ]:
response = sqs.receive_message(
    QueueUrl=queue_url,
    MaxNumberOfMessages=10,
    WaitTimeSeconds=10 
)

messages = response.get('Messages', [])

if messages:
    for message in messages:
        
        print("Message received:", message['Body'])

else:
    print("No messages received.")

#### Read Messages from the FIFO Queue

In [ ]:
response = sqs.receive_message(
    QueueUrl=fifo_queue_url,
    MaxNumberOfMessages=10,
    WaitTimeSeconds=10 
)

messages = response.get('Messages', [])

if messages:
    for message in messages:
        
        print("Message received:", message['Body'])

else:
    print("No messages received.")

#### Deletes messages to prevent reprocessing

In [ ]:
# Standard Queue

response = sqs.receive_message(
    QueueUrl=queue_url,
    MaxNumberOfMessages=10,
    WaitTimeSeconds=10 
)

messages = response.get('Messages', [])

if messages:
    for message in messages:
        
        print("Message received:", message['Body'])

        sqs.delete_message(
            QueueUrl=queue_url,
            ReceiptHandle=message['ReceiptHandle']
        )
        print("Message deleted from the queue.")

else:
    print("No messages received.")

In [ ]:
#FiFO Queue

response = sqs.receive_message(
    QueueUrl=fifo_queue_url,
    MaxNumberOfMessages=10,
    WaitTimeSeconds=10 
)

messages = response.get('Messages', [])

if messages:
    for message in messages:
        
        print("Message received:", message['Body'])

        sqs.delete_message(
            QueueUrl=fifo_queue_url,
            ReceiptHandle=message['ReceiptHandle']
        )
        print("Message deleted from the queue.")

else:
    print("No messages received.")